# Day 22 - Statistics: Mean Median Mode

This notebook covers Measures of Central Tendency, Skewness & Outlier Sensitivity, Linear Transformations, and Distributed Aggregations

## Validation goals
- Understand the mathematical definition and sensitivity of the arithmetic mean[cite: 17]
- Analyze the robustness of the median in skewed distributions and outlier-heavy datasets[cite: 17]
- Apply the mode to both categorical data and numerical distributions[cite: 17]
- Demonstrate the effects of shifting (adding $c$) and scaling (multiplying by $k$) on central tendency metrics[cite: 17]
- Evaluate the computational complexity and parallelization tradeoffs between mean ($\mathcal{O}(n)$) and median ($\mathcal{O}(n \log n)$) in distributed environments[cite: 17]

Transitioning from data manipulation in Pandas, Week 4 introduces Probability and Statistics—the mathematical foundation of machine learning, data analysis, and decision intelligence.

Before building predictive models or calculating variance, you must understand the center of your data distribution. Measures of central tendency summarize a dataset using a single representative value. While they sound basic, choosing the correct metric depends heavily on data distribution shape, skewness, and the presence of outliers.

### Core Concepts & Deep Dive1

1. **The Mean (Arithmetic Average)**
The mean is the sum of all values divided by the total number of observations ($n$).
- Mathematical Property: It balances all points perfectly, meaning the sum of deviations from the mean is always zero.
- Sensitivity to Outliers: Because every data point is weighted equally in the numerator, the mean is highly sensitive to extreme values (outliers). In right-skewed distributions (e.g., household incomes or software engineering salaries), a few extremely high values pull the mean upward, making it unrepresentative of a "typical" data point.

2. **The Median (Middle Value)**
The median is the physical middle of a dataset when sorted in ascending or descending order. If the dataset has an odd number of observations, it is the exact center element; if even, it is the arithmetic mean of the two middle elements.
- Robustness: Unlike the mean, the median is robust to outliers and skewed distributions. If a single salary in a room of 10 people changes from $50,000 to $10,000,000, the mean changes drastically, but the median remains completely unaffected.
- Computational Cost: Finding the median requires sorting the array, giving it a time complexity of $\mathcal{O}(n \log n)$ compared to the $\mathcal{O}(n)$ linear scan required for the mean.

3. The Mode (Most Frequent Value)
The mode is the value that appears most frequently in a dataset.
- Categorical vs. Numerical: The mode is the only measure of central tendency that can be applied directly to categorical (nominal/ordinal) data (e.g., product categories, error codes, customer tiers).
- Multimodal Distributions: A dataset can be unimodal, bimodal, or multimodal if multiple values share the highest frequency count.

### Calculating Central Tendency in Python

Here is a script demonstrating how to compute and analyze mean, median, and mode using NumPy and Pandas, highlighting how outliers distort the mean while leaving the median stable.

In [1]:
import numpy as np
import pandas as pd

# 1. Generate a skewed dataset representing server response times (milliseconds)
np.random.seed(42)
normal_latencies = np.random.normal(loc=45.0, scale=5.0, size=995)
# Inject 5 extreme latency spikes (outliers)
outliers = np.array([500.0, 750.0, 1200.0, 1500.0, 2500.0])
latencies = np.concatenate([normal_latencies, outliers])

df = pd.DataFrame({"response_time_ms": latencies})

# 2. Compute measures of central tendency
mean_val = df["response_time_ms"].mean()
median_val = df["response_time_ms"].median()
mode_val = df["response_time_ms"].mode()[0] # .mode() returns a Series

print(f"--- Server Latency Central Tendency ---")
print(f"Mean Response Time:   {mean_val:.2f} ms")
print(f"Median Response Time: {median_val:.2f} ms")
print(f"Mode Response Time:   {mode_val:.2f} ms")

# 3. Categorical Mode Example
categories = pd.Series(["ERROR", "INFO", "WARNING", "INFO", "INFO", "DEBUG"])
print(f"\nCategorical Mode (Log Levels): {categories.mode()[0]}")

--- Server Latency Central Tendency ---
Mean Response Time:   51.31 ms
Median Response Time: 45.14 ms
Mode Response Time:   28.79 ms

Categorical Mode (Log Levels): INFO


### Guided Question:

Suppose you are analyzing property prices in a major metropolitan area where 90% of homes cost between $200,000 and $400,000, but a few luxury penthouses cost upwards of $20,000,000.

- Will the mean property price be higher, lower, or approximately equal to the median property price?

- Why is the median typically reported by real estate analysts instead of the mean when describing housing markets?

1. The mean property price will typically be higher than the median because the distribution is positively skewed (right-skewed).
- Mean: Adds all property prices and divides by the number of properties. Extremely expensive penthouses pull the mean upward.
- Median: The middle property price when all properties are sorted from lowest to highest. A few extremely expensive properties have little effect on it.
For example, consider these five property prices:

$200,000, $250,000, $300,000, $400,000, $20,000,000
- Mean = $4,230,000
- Median = $300,000

2. The median often better represents the price of a typical home because it is resistant to extreme values (outliers).
- Less sensitive to luxury properties: A handful of multimillion-dollar homes won't substantially distort the median.
- More representative of ordinary buyers: It better describes the midpoint of the market, particularly when most properties fall within a moderate price range.
- More stable for comparisons: Median prices can make comparisons between neighborhoods or across time less sensitive to changes in a few exceptionally expensive sales.


Key takeaway: In a right-skewed housing market, the mean is usually greater than the median. Analysts often prefer the median to describe a typical property price, although the mean remains useful when studying total market value or average expenditure.

### Guided Question

1. If every single data point in a dataset is increased by a constant value $c$, how do the mean, median, and mode change?
2.  What happens to these three metrics if every data point is multiplied by a scaling factor $k$?

1. Suppose the original dataset is \([2, 4, 4, 6, 9]\), and we add \(c=3\) to every value.

The new dataset becomes \([5, 7, 7, 9, 12]\).

Metric	Original	After adding \(c\)	Change
Mean	5	        8	                Increases by 3
Median	4	        7	                Increases by 3
Mode	4	        7	                Increases by 3

**General rule:**
- Mean: \(\bar{x}_{new} = \bar{x} + c\)
- Median: \(M_{new} = M + c\)
- Mode: \(Mo_{new} = Mo + c\)

Adding a constant shifts the entire distribution without changing its shape or spread.


2. Using the same original dataset, multiply every value by \(k=2\).
The new dataset becomes \([4, 8, 8, 12, 18]\).

**General rule:**
- Mean: \(\bar{x}_{new} = k\bar{x}\)
- Median: \(M_{new} = kM\), for \(k>0\)
- Mode: \(Mo_{new} = kMo\)

Multiplication scales the location and spread of the distribution. If \(k\) is negative, the order reverses, so the median still scales by \(k\), but the mode becomes \(kMo\) and the mean also scales by \(k\).

Key takeaway:
These properties are useful in data preprocessing. For example, converting property prices from dollars to cents multiplies every value by 100, so all three metrics also multiply by 100.

### Guided Question:

When processing massive distributed datasets (e.g., billions of rows across a Spark cluster), why is computing the mean significantly easier and more computationally efficient than computing the median?

The mean is easier to compute on massive distributed datasets because it can be calculated by combining small summary statistics, whereas the exact median generally requires determining the global ordering or rank of values across the entire dataset.

1. Computing the mean: easily parallelizable
The mean is calculated as:
\[
\bar{x}=\frac{\sum_{i=1}^{n}x_i}{n}
\]

A distributed system such as Apache Spark can divide billions of rows into partitions and calculate each partition's sum and count independently.

2. Computing the median: requires global ranking
The median is the middle value after sorting the dataset.

For an exact median, a distributed system generally needs to determine which value occupies the middle rank—or the two middle ranks when the number of observations is even.

This is harder because:
- Local medians are insufficient. Averaging the medians calculated by individual workers does not generally produce the global median.
- Values must be compared across partitions. A worker's middle value might be smaller than most values on another worker.
- Data shuffling and ordering can be expensive. Exact algorithms may require partitioning, selection, data movement, or sorting to identify the global middle value.
For example, consider two workers:
- Worker A: [1, 2, 100] — local median is 2.
- Worker B: [3, 4, 5] — local median is 4.
The global dataset is [1, 2, 3, 4, 5, 100], whose median is \(3.5\).

The average of the local medians is also \(3\), not \(3.5\). Therefore, simply combining local medians does not work.

One important nuance: computing the median does not always require fully sorting every row. Distributed selection algorithms can find it without a complete sort, but exact median computation still requires more coordination than combining sums and counts in typical distributed workflows.

**Key takeaway:** 
The mean is computationally efficient because sums and counts combine naturally across workers. The exact median is more expensive because it depends on the global distribution of values, not just a few simple aggregate statistics.

For very large datasets, approximate quantile algorithms can estimate the median with much less computation and communication when exact precision is unnecessary.